# Synthetic valley: PEST++ IES with the advanced model

This notebook is a variant of [`synthetic-valley-e-pest-interface`](synthetic-valley-e-pest-interface.ipynb) and [`synthetic-valley-f-pestpp-ies`](synthetic-valley-f-pestpp-ies.ipynb) for the **advanced model** of [`synthetic-valley-b-advanced`](synthetic-valley-b-advanced.ipynb). That model has the processes the base model leaves out: a stream that routes its flow (SFR), a lake with its own water budget (LAK), an unsaturated zone (UZF), runoff moved to the lake and the stream (MVR), and multi-aquifer pumping wells (MAW). Its structure is closer to the "truth" model, but that means the model runs slower...tradeoffs...

A more complex model brings some additional benefits: the model now can assimilate **stream flow**, which may help learn more about the **predictions**, which are the same three as in the base notebooks: the groundwater discharge to the stream in the steady state, and the smallest groundwater discharge to the stream in any year of the history, and in any year of the future.

## What this notebook covers

By the end of this notebook you will be able to:

- parameterize the packages of an advanced MODFLOW 6 model with `pyemu.utils.PstFrom`,
- add predictions that are computed from the model output after each run,
- design measurement noise for heads and for a flow, with a percentage and a floor, and
- run PEST++ IES, compare the predictions with those of the base model, and map the posterior property fields.

## Imports

`sv_notebook_helpers` holds the location of the model files, `true_observations()`, which reads every observed series from the "truth" model, and the two functions used behind the scenes: `make_homogeneous()` and `stream_flux_predictions()`.

In [ ]:
%matplotlib inline
import logging
import pathlib as pl
import warnings

import flopy
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyemu
from sv_notebook_helpers import (
    ADVANCED_MODEL,
    make_homogeneous,
    stream_flux_predictions,
    true_observations,
)

warnings.filterwarnings("ignore", module="shapefile")
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("PestUtilsLib").addHandler(logging.NullHandler())

## Write the model with external files

`set_all_data_external()` writes every array and every stress period list to its own text file, so the parameters can rewrite them - this is standard practice with PEST(++). Before that, `make_homogeneous()` gives each layer one value of K, vertical anisotropy, and storage, the geometric mean of its active cells, as in the base interface; this isn't required but we do it here to make things more challenging and interesting. The UZF, SFR, and LAK inputs of this model are already the same everywhere. 

The model is run once so that the output files exist when the observations are added.

In [ ]:
name = "sv"
org_ws = pl.Path("models/sv-adv-ies-org")  # the model with external files
template_ws = pl.Path("models/sv-adv-ies-template")  # the PEST interface

sim = flopy.mf6.MFSimulation.load(
    sim_name=name, sim_ws=ADVANCED_MODEL, write_headers=False, verbosity_level=0
)
sim.set_sim_path(org_ws)
gwf = sim.get_model(name)
make_homogeneous(gwf)  # one value per layer of each property

# a looser closure than the model's, 0.0001 m, saves iterations and changes
# heads by far less than the noise
ims = sim.get_package("ims")
ims.complexity = "moderate"
ims.outer_dvclose = 1.0e-4
ims.outer_maximum = 500
ims.inner_dvclose = 1.0e-5

sim.set_all_data_external(check_data=False)
sim.write_simulation(silent=True)
success, buff = sim.run_simulation(silent=True)
assert success, "MODFLOW 6 did not terminate normally"

grid = gwf.modelgrid
idomain = gwf.dis.idomain.array
nlay = gwf.dis.nlay.data
width = 17.15 / 2.54  # figure width, inches
gwf.get_package_list()

## Start the PEST interface

As in the base interface, `PstFrom` copies the model to `template_ws`, locates the pilot points on the model grid, and counts layers, rows, and columns from 1.

In [ ]:
pf = pyemu.utils.PstFrom(
    original_d=org_ws,
    new_d=template_ws,
    remove_existing=True,
    longnames=True,
    spatial_reference=gwf.modelgrid,
    zero_based=False,
    start_datetime="1962-01-01",
    echo=False,
)

## Observations and predictions

The model writes six output csv MF6 "observation"files: the water-table and lower-aquifer heads at the 17 observation wells, the heads at the pumping and scenario wells, the stream (the flow between the aquifer and the stream, `riv-swgw`, and the flow leaving the model at the last reach, `riv-flow`), the lake (its stage and its flow with the aquifer), and the coast. Each value is named after its file, column, and time. The **water budget** that MODFLOW 6 writes to `sv-budget.csv` is added too, in group `bud`: every flow in and out of the model by package (storage, the unsaturated zone, the stream, the lake, the coast, the wells) at every output time, with the totals and the percent discrepancy. The truth has no budget, so these observations get no weight; they come back in the ensemble files, so the prior and posterior budgets can be compared.

The three **predictions** are computed after each model run by `stream_flux_predictions()` from the SFR flows, as in the base interface from the RIV flows: the groundwater discharge to the stream in the steady state, and the smallest groundwater discharge to the stream in any year of the history, and in any year of the future.

In [ ]:
obs_files = {
    "sv.gwf.wt.csv": "wt",
    "sv.gwf.aq.csv": "aq",
    "sv.gwf.scenario.csv": "scen",
    "sv.sfr.obs.csv": "sfr",
    "sv.lake.obs.csv": "lak",
    "sv.ghb.obs.csv": "coast",
}
for f, prefix in obs_files.items():
    df = pd.read_csv(org_ws / f, index_col=0)
    pf.add_observations(
        f,
        insfile=f + ".ins",
        index_cols="time",
        use_cols=list(df.columns),
        prefix=prefix,
        obsgp=[prefix] * len(df.columns),
    )

# the water budget at every output time, every flow in and out of the model
# by package, from the budget csv that MODFLOW 6 writes (BUDGETCSV in OC);
# no weight, so IES does not fit them
budget = pd.read_csv(org_ws / "sv-budget.csv", index_col=0)
pf.add_observations(
    "sv-budget.csv",
    insfile="sv-budget.csv.ins",
    index_cols="time",
    use_cols=list(budget.columns),
    prefix="bud",
    obsgp=["bud"] * len(budget.columns),  # one group
)

# the predictions: the groundwater discharge to the stream (m3/d) in the steady
# state, and the smallest in the history and in the future, computed from the SFR flows after each run
stream_flux_predictions(
    template_ws / "sv.sfr.obs.csv", out_file=template_ws / "sv.predictions.csv"
)
pf.add_py_function(
    "sv_notebook_helpers.py",
    "stream_flux_predictions('sv.sfr.obs.csv')",
    is_pre_cmd=False,
)
pf.add_observations(
    "sv.predictions.csv",
    index_cols="prediction",
    use_cols=["discharge"],
    prefix="pred",
    obsgp="pred",
)

## Geostatistical structures

Use the structure designed with the prior widget of the base interface - copy those settings over here if you want to have a more consistent comparison between the two experiments

The same structure correlates the multipliers of neighboring cells in the UZF, SFR, and GHB lists. The multipliers of each stress period are correlated in time, with a range of 2 years.

In [ ]:
pp_space = 4
grid_gs = pyemu.geostats.GeoStruct(
    variograms=pyemu.geostats.ExpVario(contribution=1.0, a=1000.0), transform="log"
)
temporal_gs = pyemu.geostats.GeoStruct(
    variograms=pyemu.geostats.ExpVario(contribution=1.0, a=7300.0), transform="log"
)

## Hydraulic conductivity, anisotropy, and storage parameters

Each layer gets pilot points and a constant on its horizontal K, its vertical anisotropy `k33`, and its specific storage `ss`, in its active cells, as in the base interface. The lake is a LAK package here, not cells of very high K, so its cells are inactive and no layer needs a zone that leaves them out.

In [ ]:
for k in range(nlay):
    layer_zone = (idomain[k] > 0).astype(int)
    layer = f"layer{k + 1}"

    # pilot points on K, k33, and ss
    for array, base, bounds in [
        ("npf_k", f"k{k + 1}pp", (0.1, 10.0)),
        ("npf_k33", f"k33{k + 1}pp", (0.2, 5.0)),
        ("sto_ss", f"ss{k + 1}pp", (0.2, 5.0)),
    ]:
        pf.add_parameters(
            f"sv.{array}_{layer}.txt",
            par_type="pilotpoints",
            par_name_base=base,
            pargp=base,
            zone_array=layer_zone,
            lower_bound=bounds[0],
            upper_bound=bounds[1],
            geostruct=grid_gs,
            pp_options={"pp_space": pp_space, "try_use_ppu": True},
        )

    # a constant on each, so the whole layer can shift together
    for array, base, bounds in [
        ("npf_k", f"k{k + 1}cn", (0.2, 5.0)),
        ("npf_k33", f"k33{k + 1}cn", (0.1, 10.0)),
        ("sto_ss", f"ss{k + 1}cn", (0.1, 10.0)),
    ]:
        pf.add_parameters(
            f"sv.{array}_{layer}.txt",
            par_type="constant",
            par_name_base=base,
            pargp=base,
            zone_array=layer_zone,
            lower_bound=bounds[0],
            upper_bound=bounds[1],
        )

## Unsaturated zone, stream, lake, coast, and runoff parameters

The advanced packages are lists, one row per cell or reach, so their parameters are multipliers on a column of the list file. `index_cols` names the columns that identify a row, and `use_cols` the column multiplied, both counted from 0.

- **UZF** (`sv.uzf_packagedata.txt`): the vertical K of the unsaturated zone, column 7, with a multiplier per cell correlated in space (`par_type="grid"`) and one for all cells; and the saturated water content, column 9, within 20%.
- **SFR** (`sv.sfr_packagedata.txt`): the streambed K, column 9, per reach and as a whole; and Manning's n, column 10.
- **LAK** (`sv.lak_connectiondata.txt`): the lakebed leakance, column 6.
- **GHB** (`sv.ghb_stress_period_data_1.txt`): the coastal conductance, column 4, per cell and as a whole.
- **MVR** (`sv.mvr_perioddata_1.txt`): the fraction of the UZF runoff moved to the lake and the stream, column 5.


In [ ]:
# UZF vertical K: per cell, correlated in space, and for all cells
pf.add_parameters(
    "sv.uzf_packagedata.txt",
    par_type="grid",
    index_cols=[0, 1, 2, 3],
    use_cols=[7],
    par_name_base="vksgr",
    pargp="vksgr",
    lower_bound=0.2,
    upper_bound=5.0,
    geostruct=grid_gs,
)
pf.add_parameters(
    "sv.uzf_packagedata.txt",
    par_type="constant",
    index_cols=[0, 1, 2, 3],
    use_cols=[7],
    par_name_base="vks",
    pargp="vks",
    lower_bound=0.1,
    upper_bound=10.0,
)
# UZF saturated water content
pf.add_parameters(
    "sv.uzf_packagedata.txt",
    par_type="constant",
    index_cols=[0, 1, 2, 3],
    use_cols=[9],
    par_name_base="thts",
    pargp="thts",
    lower_bound=0.8,
    upper_bound=1.2,
)

# SFR streambed K: per reach, correlated along the stream, and for all reaches
pf.add_parameters(
    "sv.sfr_packagedata.txt",
    par_type="grid",
    index_cols=[0, 1, 2, 3],
    use_cols=[9],
    par_name_base="rhkgr",
    pargp="rhkgr",
    lower_bound=0.2,
    upper_bound=5.0,
    geostruct=grid_gs,
)
pf.add_parameters(
    "sv.sfr_packagedata.txt",
    par_type="constant",
    index_cols=[0, 1, 2, 3],
    use_cols=[9],
    par_name_base="rhk",
    pargp="rhk",
    lower_bound=0.1,
    upper_bound=10.0,
)
# SFR Manning's n
pf.add_parameters(
    "sv.sfr_packagedata.txt",
    par_type="constant",
    index_cols=[0, 1, 2, 3],
    use_cols=[10],
    par_name_base="man",
    pargp="man",
    lower_bound=0.5,
    upper_bound=2.0,
)

# LAK bed leakance
pf.add_parameters(
    "sv.lak_connectiondata.txt",
    par_type="constant",
    index_cols=[0, 1],
    use_cols=[6],
    par_name_base="bedleak",
    pargp="bedleak",
    lower_bound=0.1,
    upper_bound=10.0,
)

# GHB coastal conductance: per cell, correlated along the coast, and for all cells
pf.add_parameters(
    "sv.ghb_stress_period_data_1.txt",
    par_type="grid",
    index_cols=[0, 1, 2],
    use_cols=[4],
    par_name_base="coastgr",
    pargp="coastgr",
    lower_bound=0.2,
    upper_bound=5.0,
    geostruct=grid_gs,
)
pf.add_parameters(
    "sv.ghb_stress_period_data_1.txt",
    par_type="constant",
    index_cols=[0, 1, 2],
    use_cols=[4],
    par_name_base="coastcond",
    pargp="coastcond",
    lower_bound=0.1,
    upper_bound=10.0,
)

# MVR fraction of the UZF runoff moved to the lake and the stream
pf.add_parameters(
    "sv.mvr_perioddata_1.txt",
    par_type="constant",
    index_cols=[0, 1, 2, 3],
    use_cols=[5],
    par_name_base="runoff",
    pargp="runoff",
    lower_bound=0.5,
    upper_bound=2.0,
)

## Parameters for each stress period

The weather changes every year, so each stress period gets its own multipliers, correlated in time by `temporal_gs` through the date of the end of the period: the UZF infiltration (column 1 of `sv.uzf_perioddata_*.txt`), potential ET (column 2), and extinction depth (column 3), and the rainfall and evaporation on the lake (rows 0 and 1 of `sv.lak_perioddata_*.txt`).

In [ ]:
perlen = sim.tdis.perioddata.array["perlen"]
dates = pd.Timestamp("1962-01-01") + pd.to_timedelta(np.cumsum(perlen), unit="D")
for kper, date in enumerate(dates):
    uzf_file = f"sv.uzf_perioddata_{kper + 1}.txt"
    lak_file = f"sv.lak_perioddata_{kper + 1}.txt"
    for use_col, base in [(1, "finf"), (2, "pet"), (3, "extdp")]:
        pf.add_parameters(
            uzf_file,
            par_type="constant",
            index_cols=[0],
            use_cols=[use_col],
            par_name_base=base,
            pargp=base,
            lower_bound=0.5,
            upper_bound=2.0,
            datetime=date,
            geostruct=temporal_gs,
        )
    for use_row, base in [(0, "lakrain"), (1, "lakevap")]:
        pf.add_parameters(
            lak_file,
            par_type="constant",
            index_cols=[0, 1],
            use_cols=[2],
            use_rows=[use_row],
            par_name_base=base,
            pargp=base,
            lower_bound=0.5,
            upper_bound=2.0,
            datetime=date,
            geostruct=temporal_gs,
        )

## Property observations

The K, k33, and ss arrays that the forward run writes are added as observations too, one for each active cell of each layer, so that every realization's properties come back in the ensemble files and can be visualized without running the model again. They get no weight, so IES does not fit them. Specific yield is left out: every cell of the model is confined (`ICONVERT` is 0), so MODFLOW 6 does not use it, and it has no parameters.

In [ ]:
# the property arrays of every layer, as observations with no weight
for k in range(nlay):
    layer_zone = (idomain[k] > 0).astype(int)
    layer = f"layer{k + 1}"
    for array, prefix in [
        ("npf_k", f"k{k + 1}"),
        ("npf_k33", f"k33{k + 1}"),
        ("sto_ss", f"ss{k + 1}"),
    ]:
        pf.add_observations(
            f"sv.{array}_{layer}.txt",
            prefix=prefix,
            obsgp=prefix,
            zone_array=layer_zone,
        )

## Build the control file

The forward run applies the multipliers, runs `mf6`, and then computes the predictions.

In [ ]:
pf.mod_sys_cmds.append("mf6")
pst = pf.build_pst("pest.pst")
par = pst.parameter_data
print(f"{pst.npar} parameters in {len(pst.par_groups)} groups, {pst.nobs} observations")
par.pargp.value_counts().sort_index()

## Observed values and predictions

The observed values come from the truth, by each observation's column and time. The **history** is the first 11 output times, through the end of stress period 11 (day 19,723): the heads at the observation wells, as in the base interface, and now also the stream flow leaving the model, `riv-flow`. MODFLOW 6 reports flow out of the model as negative. The steady-state heads, at the first output time, are put in a group of their own, `ssheads`, as in the base interface. The weights are set below, after the noise is designed. The true predictions come from the truth's flow between the aquifer and the stream, and the option `forecasts` tells PEST++ which observations they are.

In [ ]:
truth = true_observations()
obs = pst.observation_data
obs["time"] = obs.time.astype(float)
series = obs.oname.isin(list(obs_files.values()))  # not the predictions
obs.loc[series, "obsval"] = [
    truth.loc[t, u] for u, t in zip(obs.usecol[series], obs.time[series])
]
# the steady-state heads are a group of their own, so that phi reports them apart
steady = obs.oname.isin(["wt", "aq"]) & (obs.time == 16071.0)
obs.loc[steady, "obgnme"] = "ssheads"
head_history = obs.oname.isin(["wt", "aq"]) & (obs.time <= 19723.0)
flow_history = (obs.usecol == "riv-flow") & (obs.time <= 19723.0)
obs["weight"] = 0.0

# the true predictions: the discharge to the stream in the steady state, the
# first output time, and the smallest discharge in the history and in the future
true_discharge = -truth["riv-swgw"]  # the truth's flow is negative to the stream
predictions = obs.loc[obs.oname == "pred"]
for name, prediction in zip(predictions.obsnme, predictions.prediction):
    if prediction == "steadystate":
        obs.loc[name, "obsval"] = true_discharge.iloc[0]
    elif prediction == "minhistory":
        obs.loc[name, "obsval"] = true_discharge.loc[:19723.0].min()
    else:
        obs.loc[name, "obsval"] = true_discharge.loc[19723.0:].iloc[1:].min()
pst.pestpp_options["forecasts"] = ",".join(predictions.obsnme)
print(f"{head_history.sum()} head and {flow_history.sum()} flow observations")
obs.loc[predictions.obsnme, ["prediction", "obsval"]]

## Design the noise

The noise of the heads is designed as in the base interface: a standard deviation `sd` in meters, and an exponential correlation in time with a range `a` in days, so a well that reads high one year tends to read high the next.

A flow is measured with an error that grows with the flow, so the noise of the stream flow is a **percentage** of the flow, with a **floor** in m³/d that keeps the smallest flows from being trusted too much: its standard deviation in each year is the larger of `percent` of that year's flow and `floor`. The error of a stream gauge also persists, from a rating curve that is off for years at a time, so the flow noise has its own range `flow_a` in days. The smallest value of both range sliders is the shortest stress period, because the noise is drawn at the output times, one per stress period.

The first widget shows the true heads at four wells, in red, with five noise realizations added to each, in gray; the second the true stream outflow, with five noise realizations, and the band of one standard deviation either way, dashed. The random numbers stay the same, so each slider shows what it does to the same draws.

In [ ]:
times = np.sort(obs.loc[head_history, "time"].unique())
years = (times - times[0]) / 365.25
wells = ["wt01-i2-j17", "wt05-i14-j11", "wt09-i19-j11", "wt14-i30-j6"]
min_perlen = perlen.min()  # the shortest stress period length (days)


def correlation(a):
    # the correlation of the history times, an exponential variogram of range a
    gs = pyemu.geostats.GeoStruct(
        variograms=pyemu.geostats.ExpVario(contribution=1.0, a=a)
    )
    cov = gs.covariance_matrix(
        times, np.zeros_like(times), names=[str(t) for t in times]
    )
    return cov.x


def show_head_noise(sd=0.1, a=730.0):
    rng = np.random.default_rng(2)
    fig, axes = plt.subplots(
        2, 2, figsize=(width, 0.7 * width), layout="constrained", sharex=True
    )
    for ax, well in zip(axes.flat, wells):
        # the true heads of this well, in time order
        true = obs.loc[head_history & (obs.usecol == well)].sort_values("time")
        true = true.obsval.values
        # five noise series, independent of the other wells
        noise = sd * rng.multivariate_normal(
            np.zeros(len(times)), correlation(a), size=5
        )
        for series in noise:
            ax.plot(years, true + series, color="0.6", lw=0.8)
        ax.plot(years, true, color="red", lw=1.5)
        ax.set_ylim(true.mean() - 1.5, true.mean() + 1.5)
        ax.set_title(well.split("-")[0].upper())
    axes[0, 0].set_ylabel("Head (m)")
    axes[1, 0].set_ylabel("Head (m)")
    axes[1, 0].set_xlabel("Years since the steady state")
    axes[1, 1].set_xlabel("Years since the steady state")
    plt.show()


def show_flow_noise(percent=10.0, floor=500.0, flow_a=730.0):
    rng = np.random.default_rng(3)
    flow = obs.loc[flow_history].sort_values("time")
    outflow = -flow.obsval.values  # the flow leaving the model, positive
    sd = np.maximum(percent / 100.0 * outflow, floor)
    # five noise series, correlated in time and scaled by each year's sd
    noise = sd * rng.multivariate_normal(
        np.zeros(len(times)), correlation(flow_a), size=5
    )
    fig, ax = plt.subplots(figsize=(width, 0.4 * width), layout="constrained")
    for series in noise:
        ax.plot(years, outflow + series, color="0.6", lw=0.8)
    ax.plot(years, outflow, color="red", lw=1.5)
    ax.set_ylim(0.0, 1.5 * outflow.max())
    ax.set_title("Stream outflow")
    ax.set_xlabel("Years since the steady state")
    ax.set_ylabel("Flow (m$^3$/d)")
    plt.show()


head_controls = {
    "sd": widgets.FloatSlider(value=0.1, min=0.02, max=0.5, step=0.02),
    "a": widgets.FloatSlider(
        value=2.0 * min_perlen,
        min=min_perlen,
        max=30.0 * min_perlen,
        step=min_perlen / 10.0,
    ),
}
flow_controls = {
    "percent": widgets.FloatSlider(value=10.0, min=1.0, max=50.0, step=1.0),
    "floor": widgets.FloatSlider(value=500.0, min=0.0, max=5000.0, step=100.0),
    "flow_a": widgets.FloatSlider(
        value=2.0 * min_perlen,
        min=min_perlen,
        max=30.0 * min_perlen,
        step=min_perlen / 10.0,
    ),
}
for controls in [head_controls, flow_controls]:
    for name, control in controls.items():
        control.description = name
head_controls["a"].description = "a (days)"
flow_controls["flow_a"].description = "flow_a (days)"
display(
    widgets.HBox(list(head_controls.values())),
    widgets.interactive_output(show_head_noise, head_controls),
    widgets.HBox(list(flow_controls.values())),
    widgets.interactive_output(show_flow_noise, flow_controls),
)

The next cell takes the final values of the sliders and sets the standard deviation and the weight (its inverse) of every history observation; rerun it, and the cells after it, after moving a slider.

In [ ]:
# the final values of the widgets' sliders
head_sd = head_controls["sd"].value
head_a = head_controls["a"].value
flow_percent = flow_controls["percent"].value
flow_floor = flow_controls["floor"].value
flow_a = flow_controls["flow_a"].value

obs.loc[head_history, "standard_deviation"] = head_sd
outflow = -obs.loc[flow_history, "obsval"]
obs.loc[flow_history, "standard_deviation"] = np.maximum(
    flow_percent / 100.0 * outflow, flow_floor
)
history = head_history | flow_history
obs.loc[history, "weight"] = 1.0 / obs.loc[history, "standard_deviation"]
obs.loc[obs.obgnme == "bud", "weight"] = 0.0  # the budget is not observed
print(f"heads: sd {head_sd} m, range {head_a} days")
print(f"flow: {flow_percent}% with a floor of {flow_floor} m3/d, range {flow_a} days")
obs.loc[flow_history, ["time", "obsval", "standard_deviation", "weight"]]

## Draw the noise ensemble

`pyemu.helpers.autocorrelated_draw()` takes one geostatistical structure for each series, correlates the noise of that series in time, and scales it by each observation's standard deviation, so the flow noise grows with the flow. Each well's heads, and the stream flow, are separate structures, so their noise is independent of each other.

In [ ]:
num_reals = 500
struct_dict = {}
for well, df in obs.loc[head_history].groupby("usecol"):
    well_gs = pyemu.geostats.GeoStruct(
        variograms=pyemu.geostats.ExpVario(contribution=1.0, a=head_a), name=well
    )
    struct_dict[well_gs] = df.obsnme.tolist()
flow_gs = pyemu.geostats.GeoStruct(
    variograms=pyemu.geostats.ExpVario(contribution=1.0, a=flow_a), name="riv-flow"
)
struct_dict[flow_gs] = obs.loc[flow_history, "obsnme"].tolist()
oe = pyemu.helpers.autocorrelated_draw(
    pst, struct_dict, time_distance_col="time", num_reals=num_reals, verbose=False
)
oe.to_csv(template_ws / "obs_noise.csv")

flow = obs.loc[flow_history].sort_values("time")
years = (flow.time - flow.time.min()) / 365.25
with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(figsize=(width, 0.4 * width), layout="constrained")
    ax.plot(years, -oe.loc[:, flow.obsnme].values.T, color="0.7", lw=0.5)
    ax.plot(years, -flow.obsval, "o-", color="red", ms=4, lw=1.0, label="True")
    flopy.plot.styles.xlabel(ax=ax, label="Years since the steady state")
    flopy.plot.styles.ylabel(ax=ax, label="Stream outflow (m$^3$/d)")
    flopy.plot.styles.heading(ax=ax, heading="Stream outflow: noise realizations")
    flopy.plot.styles.graph_legend(ax=ax, loc="lower right")

## Draw the prior parameter ensemble

`pf.draw()` draws the parameter ensemble from the bounds and the geostatistical structures, and `enforce()` clips any draw that falls outside its bounds. The plot shows the infiltration multiplier of every stress period in every realization, with five of them picked out: consecutive years are correlated by `temporal_gs`.

In [ ]:
pe = pf.draw(num_reals=num_reals, use_specsim=True)
pe.enforce()
pe.to_csv(template_ws / "prior_pe.csv")

finf = par.loc[par.pargp == "finf"]
kper = finf.inst.astype(int) + 1  # the instance counts the add_parameters() calls
with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(figsize=(width, 0.4 * width), layout="constrained")
    ax.plot(kper, pe.loc[:, finf.parnme].values.T, color="0.85", lw=0.5)
    ax.plot(kper, pe.loc[pe.index[:5], finf.parnme].values.T, marker="o", ms=3, lw=1.0)
    ax.axhline(1.0, color="red", lw=1.0)
    ax.set_yscale("log")
    flopy.plot.styles.xlabel(ax=ax, label="Stress period")
    flopy.plot.styles.ylabel(ax=ax, label="Infiltration multiplier")

## PEST++ IES settings and a test run
`noptmax=0` makes one run with the initial parameter values, which checks the forward run, the predictions, and the instruction files.

In [ ]:
num_workers = 25
num_reals = 50
noptmax = 5

In [ ]:
pst.pestpp_options["ies_parameter_ensemble"] = "prior_pe.csv"
pst.pestpp_options["ies_observation_ensemble"] = "obs_noise.csv"
pst.pestpp_options["ies_num_reals"] = num_reals
pst.pestpp_options["ies_subset_size"] = 10
pst.pestpp_options["ies_bad_phi_sigma"] = 1.7
pst.pestpp_options["overdue_giveup_fac"] = 1.0e30  # never kill slow runs
pst.control_data.noptmax = 0
pst.write(template_ws / "pest.pst", version=2)

pyemu.os_utils.run("pestpp-ies pest.pst", cwd=template_ws)
pst = pyemu.Pst(str(template_ws / "pest.pst"))
print(f"phi of the homogeneous model {pst.phi:.0f} for {pst.nnz_obs} observations")
pst.phi_components

`phi_components` splits phi by group, which shows how much of the misfit of the homogeneous model is in the heads and how much in the stream flow.

Set the number of iterations and write the control file.

In [ ]:
pst.control_data.noptmax = noptmax
pst.write(template_ws / "pest.pst", version=2)

## Run PEST++ IES

`start_workers()` runs IES in parallel, as in the companion notebook. A run of the advanced model takes longer than a run of the base model, so each iteration does too.

In [ ]:
m_d = pl.Path("models/sv-adv-ies-master")

In [ ]:
pyemu.os_utils.start_workers(
    str(template_ws),
    "pestpp-ies",
    "pest.pst",
    num_workers=num_workers,
    worker_root="models",
    master_dir=str(m_d),
    verbose=False,
)

In [ ]:
phi = pd.read_csv(m_d / "pest.phi.actual.csv", index_col=0)
reals = phi.columns[5:]  # the realization columns follow the summary columns
with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(figsize=(width, 0.45 * width), layout="constrained")
    ax.plot(phi.index, phi.loc[:, reals], color="0.6", lw=0.5)
    ax.plot(phi.index, phi["mean"], color="black", lw=1.5, label="Ensemble mean")
    ax.axhline(pst.nnz_obs, color="red", lw=1.0, label="Number of observations")
    ax.set_yscale("log")
    ax.set_xticks(phi.index)
    flopy.plot.styles.xlabel(ax=ax, label="Iteration")
    flopy.plot.styles.ylabel(ax=ax, label="Phi")
    flopy.plot.styles.graph_legend(ax=ax, loc="upper right")
print(phi[["mean", "min", "max"]].round(0))

## Stream outflow and heads: prior and posterior

The prior (gray) and posterior (blue) realizations of the stream outflow and of the heads at two wells, with the truth in red, through the history. Set `zoom = True` to zoom the y axis to the posterior and the noisy observations, or `False` to see the whole prior range.

In [ ]:
# True: zoom the y axis to the posterior and the noisy observations;
# False: show the whole prior range
zoom = True

noptmax = pst.control_data.noptmax
prior_oe = pd.read_csv(m_d / "pest.0.obs.csv", index_col=0)
post_oe = pd.read_csv(m_d / f"pest.{noptmax}.obs.csv", index_col=0)
noise_oe = pd.read_csv(m_d / "pest.obs+noise.csv", index_col=0)


def series_years(usecol):
    history = (obs.usecol == usecol) & (obs.time <= 19723.0)
    series = obs.loc[history].sort_values("time")
    return series, (series.time - series.time.min()) / 365.25


def zoom_to_posterior(ax, names, sign=1.0):
    """set the y range to the posterior and the noisy observations, with a buffer"""
    nz = [name for name in names if obs.loc[name, "weight"] > 0]
    values = np.concatenate(
        [
            sign * post_oe.loc[:, names].values.ravel(),
            sign * noise_oe.loc[:, nz].values.ravel(),
            sign * obs.loc[names, "obsval"].values,
        ]
    )
    buffer = 0.1 * (values.max() - values.min())
    ax.set_ylim(values.min() - buffer, values.max() + buffer)


with flopy.plot.styles.USGSPlot():
    fig, (ax1, ax2, ax3) = plt.subplots(
        3, 1, figsize=(width, 1.0 * width), layout="constrained", sharex=True
    )
    flow, years = series_years("riv-flow")
    ax1.plot(
        years, -noise_oe.loc[:, flow.obsnme].values.T, color="red", lw=0.5, alpha=0.5
    )
    ax1.plot(
        years, -prior_oe.loc[:, flow.obsnme].values.T, color="0.7", lw=0.5, alpha=0.5
    )
    ax1.plot(
        years,
        -post_oe.loc[:, flow.obsnme].values.T,
        color="tab:blue",
        lw=0.5,
        alpha=0.5,
    )

    ax1.scatter(years, -flow.obsval, color="red", marker=".", zorder=10)
    if zoom:
        zoom_to_posterior(ax1, flow.obsnme.tolist(), sign=-1.0)
    flopy.plot.styles.heading(ax=ax1, letter="A", heading="Stream outflow")
    flopy.plot.styles.ylabel(ax=ax1, label="Flow (m$^3$/d)")

    head, years = series_years("wt05-i14-j11")
    ax2.plot(years, noise_oe.loc[:, head.obsnme].values.T, color="r", lw=0.5, alpha=0.5)
    ax2.plot(
        years, prior_oe.loc[:, head.obsnme].values.T, color="0.7", lw=0.5, alpha=0.5
    )
    ax2.plot(
        years, post_oe.loc[:, head.obsnme].values.T, color="tab:blue", lw=0.5, alpha=0.5
    )
    ax2.scatter(years, head.obsval, color="red", marker=".", zorder=10)
    if zoom:
        zoom_to_posterior(ax2, head.obsnme.tolist())
    flopy.plot.styles.heading(ax=ax2, letter="B", heading="wt05-i14-j11")
    flopy.plot.styles.ylabel(ax=ax2, label="Head (m)")

    head, years = series_years("aq13-i28-j23")
    ax3.plot(years, noise_oe.loc[:, head.obsnme].values.T, color="r", lw=0.5, alpha=0.5)
    ax3.plot(
        years, prior_oe.loc[:, head.obsnme].values.T, color="0.7", lw=0.5, alpha=0.5
    )
    ax3.plot(
        years, post_oe.loc[:, head.obsnme].values.T, color="tab:blue", lw=0.5, alpha=0.5
    )
    ax3.scatter(years, head.obsval, color="red", marker=".", zorder=10)
    if zoom:
        zoom_to_posterior(ax3, head.obsnme.tolist())
    flopy.plot.styles.heading(ax=ax3, letter="C", heading="aq13-i28-j23")
    flopy.plot.styles.ylabel(ax=ax3, label="Head (m)")
    flopy.plot.styles.xlabel(ax=ax3, label="Years since the steady state")

## Steady-state heads: one-to-one plot

The steady-state heads, group `ssheads`, of every realization against the observed heads, one small dot for each head of each realization: the prior in gray, the posterior in blue, and in red the noisy observations each realization was matched to, `pest.obs+noise.csv`. The larger blue dots are the posterior medians, and the box gives the statistics of their residuals.

In [ ]:
ssheads = obs.loc[obs.obgnme == "ssheads"]
observed = ssheads.obsval
median = post_oe.loc[:, observed.index].median()
residual = observed - median
low = observed.min() - 1.0
high = observed.max() + 1.0

fig, ax = plt.subplots(figsize=(6.0, 6.0), layout="constrained")
ax.plot([low, high], [low, high], color="black", lw=1.0)
for real in prior_oe.index:
    ax.plot(observed, prior_oe.loc[real, observed.index], ".", color="0.6", ms=2)
for real in noise_oe.index:
    ax.plot(observed, noise_oe.loc[real, observed.index], ".", color="red", ms=2)
for real in post_oe.index:
    ax.plot(observed, post_oe.loc[real, observed.index], ".", color="tab:blue", ms=2)
ax.plot([], [], ".", color="0.6", label="Prior")
ax.plot([], [], ".", color="red", label="Observed + noise")
ax.plot([], [], ".", color="tab:blue", label="Posterior")
ax.plot(observed, median, "o", color="tab:blue", ms=5, label="Posterior median")
ax.set_xlim(low, high)
ax.set_ylim(low, high)
ax.set_aspect("equal")
ax.set_xlabel("Observed head (m)")
ax.set_ylabel("Simulated head (m)")
ax.legend(loc="lower right")
weighted = ssheads.weight * residual
stats = (
    f"observations {len(residual)}\n"
    f"mean residual {residual.mean():.3f} m\n"
    f"RMS residual {np.sqrt((residual**2).mean()):.3f} m\n"
    f"largest |residual| {residual.abs().max():.3f} m\n"
    f"phi of the median {(weighted**2).sum():.1f}"
)
ax.text(0.03, 0.97, stats, transform=ax.transAxes, va="top", family="monospace")
plt.show()

## Property fields

Every property array is also an observation: the forward run writes the K, vertical K, and specific storage of each layer to a file, and the instruction files read one value per active cell (groups `k1`, …, `k335`, …, `ss5`, with the row `i` and column `j` of each cell). So the observation ensembles already hold the property fields of every realization, and no extra model runs are needed to map them.

The maps show, for each layer, the posterior mean of log10 of the property, the `base` realization, which started from the homogeneous model, and three other posterior realizations. Each row, a layer, has its own color scale, so the variation within each layer shows even where the layers differ by orders of magnitude. The blank cells are inactive, including the lake in layer 1. The mean is smoother than any one realization: the realizations differ where the history says little, and averaging them cancels those differences.

In [ ]:
nlay, nrow, ncol = gwf.dis.nlay.data, gwf.dis.nrow.data, gwf.dis.ncol.data
post_reals = [real for real in post_oe.index if real != "base"][:3]
titles = ["Posterior mean", "Base"] + [f"Realization {real}" for real in post_reals]


def property_maps(prefix, label):
    # a table per layer: one row per panel, one column per active cell
    tables = []
    for lay in range(nlay):
        arr_obs = obs.loc[obs.oname == f"{prefix}{lay + 1}"]
        log_values = np.log10(post_oe.loc[:, arr_obs.obsnme].astype(float))
        mean = log_values.mean().to_frame("mean").T
        tables.append(
            (arr_obs, pd.concat([mean, log_values.loc[["base"] + post_reals]]))
        )
    with flopy.plot.styles.USGSMap():
        fig, axes = plt.subplots(
            nlay, len(titles), figsize=(width, 1.2 * width), layout="constrained"
        )
        for lay, (arr_obs, table) in enumerate(tables):
            i, j = arr_obs.i.astype(int).values, arr_obs.j.astype(int).values
            vmin, vmax = np.percentile(table.values, [2, 98])  # a scale per layer
            for col in range(len(titles)):
                ax = axes[lay, col]
                values = np.full((nrow, ncol), np.nan)
                values[i, j] = table.iloc[col].values
                mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=lay)
                pa = mm.plot_array(
                    np.ma.masked_invalid(values), cmap="viridis", vmin=vmin, vmax=vmax
                )
                ax.set_aspect("equal")
                ax.set_xticks([])
                ax.set_yticks([])
                if lay == 0:
                    ax.set_title(titles[col], fontsize=7)
                if col == 0:
                    ax.set_ylabel(f"Layer {lay + 1}", fontsize=8)
            cb = fig.colorbar(pa, ax=axes[lay, :], shrink=0.9, aspect=10)
            cb.ax.tick_params(labelsize=7)
        fig.suptitle(label, fontsize=9)


property_maps("k", "log$_{10}$ K (m/d)")

In [ ]:
property_maps("k33", "log$_{10}$ vertical K (m/d)")

In [ ]:
property_maps("ss", "log$_{10}$ specific storage (1/m)")

## Predictions

The three predictions, the groundwater discharge to the stream in the steady state (A) and the smallest in any year of the history (B) and of the future (C), prior (gray) and posterior (blue), with the true value in red. Compare them with the same predictions of the base model in [`synthetic-valley-f-pestpp-ies`](synthetic-valley-f-pestpp-ies.ipynb).

In [ ]:
steady_name = predictions.loc[predictions.prediction == "steadystate"].obsnme.iloc[0]
history_name = predictions.loc[predictions.prediction == "minhistory"].obsnme.iloc[0]
future_name = predictions.loc[predictions.prediction == "minfuture"].obsnme.iloc[0]

with flopy.plot.styles.USGSPlot():
    fig, (ax1, ax2, ax3) = plt.subplots(
        1, 3, figsize=(width, 0.4 * width), layout="constrained", sharey=True
    )
    ax1.hist(prior_oe.loc[:, steady_name], bins=15, color="0.7", alpha=0.7)
    ax1.hist(post_oe.loc[:, steady_name], bins=15, color="tab:blue", alpha=0.7)
    ax1.axvline(obs.loc[steady_name, "obsval"], color="red", lw=1.5)
    flopy.plot.styles.heading(ax=ax1, letter="A", heading="Steady state")
    flopy.plot.styles.xlabel(ax=ax1, label="Discharge (m$^3$/d)")
    flopy.plot.styles.ylabel(ax=ax1, label="Realizations")

    ax2.hist(prior_oe.loc[:, history_name], bins=15, color="0.7", alpha=0.7)
    ax2.hist(post_oe.loc[:, history_name], bins=15, color="tab:blue", alpha=0.7)
    ax2.axvline(obs.loc[history_name, "obsval"], color="red", lw=1.5)
    flopy.plot.styles.heading(ax=ax2, letter="B", heading="Smallest, history")
    flopy.plot.styles.xlabel(ax=ax2, label="Discharge (m$^3$/d)")

    ax3.hist(
        prior_oe.loc[:, future_name], bins=15, color="0.7", alpha=0.7, label="Prior"
    )
    ax3.hist(
        post_oe.loc[:, future_name],
        bins=15,
        color="tab:blue",
        alpha=0.7,
        label="Posterior",
    )
    ax3.axvline(obs.loc[future_name, "obsval"], color="red", lw=1.5, label="Truth")
    flopy.plot.styles.heading(ax=ax3, letter="C", heading="Smallest, future")
    flopy.plot.styles.xlabel(ax=ax3, label="Discharge (m$^3$/d)")
    flopy.plot.styles.graph_legend(ax=ax3, loc="upper right")